In [1]:
import kagglehub
import os
import cv2 as cv
import numpy as np
from ultralytics import YOLO
path = kagglehub.dataset_download("youssefsalahzakria/fruit-and-vegetables-classification")

train_dir = os.path.join(path, "train")
test_dir = os.path.join(path, "test")
print(f"Training directory: {train_dir}")
print(f"Test directory: {test_dir}")

Training directory: /home/lilgreg/.cache/kagglehub/datasets/youssefsalahzakria/fruit-and-vegetables-classification/versions/5/train
Test directory: /home/lilgreg/.cache/kagglehub/datasets/youssefsalahzakria/fruit-and-vegetables-classification/versions/5/test


In [2]:
model = YOLO("yolo11n-cls.pt")
results = model.train(
    data=train_dir, 
    epochs=10, 
    imgsz=128,
    hsv_s=0.5,   # Randomly vary saturation by 10%
    hsv_v=0.4,   # Randomly vary brightness by 40%
    degrees=20,  
    flipud=0.5,
    scale=0.5,   # Randomly zoom 50%-150%
    translate=0.3 ,  # Random translation to shift position further
    perspective = 0.0005,
    project="runs/fruit_veggie",  
    name="training_v5" 
)

New https://pypi.org/project/ultralytics/8.4.38 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.36 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2060, 5739MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/lilgreg/.cache/kagglehub/datasets/youssefsalahzakria/fruit-and-vegetables-classification/versions/5/train, degrees=20, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=128, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0,

In [3]:
def predict_frame(model, frame):
    h, w, _ = frame.shape
    size = min(h, w) # Get the smallest dimension to make a square
    
    # Calculate coordinates for a center crop
    start_x = (w - size) // 2
    start_y = (h - size) // 2
    
    # Crop the image to just the center square
    cropped_frame = frame[start_y:start_y+size, start_x:start_x+size]

    zoomed_frame = cv.resize(cropped_frame, (224, 224), interpolation=cv.INTER_LINEAR)
    
    # Optional: Draw a box on the original frame so you know where to hold the fruit
    cv.rectangle(frame, (start_x, start_y), (start_x+size, start_y+size), (255, 0, 0), 2)

    results = model.predict(zoomed_frame, verbose=False)
    
    if results and len(results) > 0:
        probs = results[0].probs
        return results[0].names[probs.top1], probs.top1conf.item()
    return "None", 0.0

def boost_vibrancy(frame):
    # 1. Convert to HSV (Hue, Saturation, Value)
    hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV).astype("float32")
    
    # 2. Boost Saturation (the middle channel) by 1.5x
    hsv[:, :, 1] = hsv[:, :, 1] * 1.7
    hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 255) # Keep in 0-255 range
    
    # 3. Convert back to BGR
    vibrant_frame = cv.cvtColor(hsv.astype("uint8"), cv.COLOR_HSV2BGR)
    
    # 4. Light Contrast Boost (Alpha > 1.0)
    vibrant_frame = cv.convertScaleAbs(vibrant_frame, alpha=1.5, beta=10)
    
    return vibrant_frame

In [5]:
##
# Opens a video capture device with a resolution of 800x600
# at 30 FPS.
##
def open_camera(cam_id = 0):
    # Specify the backend (CAP_V4L2) for Linux
    cap = cv.VideoCapture(cam_id, cv.CAP_V4L2)
    
    # Check if it actually opened
    if not cap.isOpened():
        print(f"Could not open camera {cam_id}. Trying index 1...")
        cap = cv.VideoCapture(1, cv.CAP_V4L2) # Sometimes the ID shifts to 1
        
    cap.set(cv.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv.CAP_PROP_FRAME_HEIGHT, 480)
    return cap
 
##
# Gets a frame from an open video device, or returns None
# if the capture could not be made.
##
def get_frame(device):
    ret, img = device.read()
    if (ret == False): # failed to capture
        print("Failed to capture from camera. Check if it is connected and try again.")
        return None
    return img
 
##
# Closes all OpenCV windows and releases video capture device
# before exit.
##
def cleanup(device): 
    cv.destroyAllWindows()
    device.release()
 

########### Main Program ###########
if __name__ == "__main__":
    # 1. Load the model here!
    model = YOLO("runs/classify/runs/fruit_veggie/training_v52/weights/best.pt")
    
    camera_id = 0
    dev = open_camera(camera_id)

    while True:
        img_orig = get_frame(dev)
        if img_orig is None:
            break
        
        # Get prediction
        img_vibrant = boost_vibrancy(img_orig)
        disp_img = img_vibrant
        pred_class, confidence = predict_frame(model, disp_img)

        # 2. Only display if confidence is high (e.g., > 50%)
        if confidence > 0.50:
            label = f"{pred_class} ({confidence*100:.1f}%)"
            # Draw the text on the image
            cv.putText(disp_img, label, (50, 50), cv.FONT_HERSHEY_SIMPLEX, 
                       1, (0, 255, 0), 2, cv.LINE_AA)
        
        cv.imshow("Fruit & Veg Detection", disp_img)

        if cv.waitKey(1) == 27: # ESC to quit
            break
 
    cleanup(dev)

QObject::moveToThread: Current thread (0x604e8d40) is not the object's thread (0x265d4940).
Cannot move to target thread (0x604e8d40)

QObject::moveToThread: Current thread (0x604e8d40) is not the object's thread (0x265d4940).
Cannot move to target thread (0x604e8d40)

QObject::moveToThread: Current thread (0x604e8d40) is not the object's thread (0x265d4940).
Cannot move to target thread (0x604e8d40)

QObject::moveToThread: Current thread (0x604e8d40) is not the object's thread (0x265d4940).
Cannot move to target thread (0x604e8d40)

QObject::moveToThread: Current thread (0x604e8d40) is not the object's thread (0x265d4940).
Cannot move to target thread (0x604e8d40)

QObject::moveToThread: Current thread (0x604e8d40) is not the object's thread (0x265d4940).
Cannot move to target thread (0x604e8d40)

QObject::moveToThread: Current thread (0x604e8d40) is not the object's thread (0x265d4940).
Cannot move to target thread (0x604e8d40)

QObject::moveToThread: Current thread (0x604e8d40) is n